In [31]:
import os
import msal
import requests

CLIENT_ID = os.environ.get("EMAIL_AGENT_ID")

SCOPES = ["Mail.Read", "Mail.ReadWrite", "Mail.Send", "User.Read"]
AUTHORITY = "https://login.microsoftonline.com/common"
CACHE_FILE = "../token_cache.bin"

In [32]:
CLIENT_ID

'f0eba5f8-4fef-42cd-b942-0f2337d6df1d'

In [33]:
os.environ.get("MICROSOFT_ID")

'fed2b938-1432-445a-95e3-a564d33272c8'

In [34]:
if not CLIENT_ID:
    raise SystemExit("CLIENT_ID 环境变量不存在，请检查对应拼写")

In [35]:
cache = msal.SerializableTokenCache()
if os.path.exists(CACHE_FILE):
    with open(CACHE_FILE, encoding="utf-8") as f:
        cache.deserialize(f.read())

In [36]:
app = msal.PublicClientApplication(CLIENT_ID, authority=AUTHORITY, token_cache=cache)

In [37]:
def get_token():
    account= app.get_accounts()
    if account:
        result = app.acquire_token_silent(SCOPES, account=account[0])
        if result and "access_token" in result:
            return result["access_token"]

    flow = app.initiate_device_flow(scopes=SCOPES)
    if "error" in flow:
        raise RuntimeError(flow["error_description"])

    print(flow["message"])
    result = app.acquire_token_by_device_flow(flow)

    if "access_token" in result:
        return result["access_token"]
    raise RuntimeError(f'{result.get("error")}: {result.get("error_description")}')

In [39]:
headers = {"Authorization" : f"Bearer {get_token()}"}
resp = requests.get("https://graph.microsoft.com/v1.0/me", headers=headers)
print(resp.status_code, resp.json())

To sign in, use a web browser to open the page https://login.microsoft.com/device and enter the code NLQJBS3FY to authenticate.
200 {'@odata.context': 'https://graph.microsoft.com/v1.0/$metadata#users/$entity', 'userPrincipalName': '799119163@qq.com', 'id': 'efd107f48a76b0bb', 'displayName': 'jiarui rao', 'surname': 'rao', 'givenName': 'jiarui', 'preferredLanguage': 'zh-CN', 'mail': '799119163@qq.com', 'mobilePhone': None, 'jobTitle': None, 'officeLocation': None, 'businessPhones': []}


In [ ]:
token = get_token()
if cache.has_state_changed:
    with open(CACHE_FILE, "w", encoding="uft-8") as f:
        # 更新缓存
        f.write(cache.serialize())
    print("OK, token 前20位: " + token[:20] + "...")